# Feature Engineering & Data Merging

In this notebook, we will merge the cleaned Traffy Fondue dataset with external data sources (PM2.5, Rainfall, and Department locations) to create a comprehensive dataset for analysis and modeling.

**Goal:** Combine all available data and select important features.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Define file paths
traffy_path = "../../data/processed/bangkok_traffy_sample_300000.csv"
pm_path =  "../scraping/pm25_data.csv"
department_path =  "../scraping/department_data.csv"
rainfall_path =  "../scraping/rainfall_data.csv"

# Load data
print("Loading datasets...")
df_traffy = pd.read_csv(traffy_path)
df_pm = pd.read_csv(pm_path)
df_dept = pd.read_csv(department_path)
df_rain = pd.read_csv(rainfall_path)

print(f"Traffy shape: {df_traffy.shape}")
print(f"PM2.5 shape: {df_pm.shape}")
print(f"Department shape: {df_dept.shape}")
print(f"Rainfall shape: {df_rain.shape}")

Loading datasets...
Traffy shape: (300000, 20)
PM2.5 shape: (912, 7)
Department shape: (286, 4)
Rainfall shape: (1492, 6)
Traffy shape: (300000, 20)
PM2.5 shape: (912, 7)
Department shape: (286, 4)
Rainfall shape: (1492, 6)


In [ ]:
# 1. Preprocessing Dates for Merging

# Convert timestamps to datetime objects
df_traffy['timestamp'] = pd.to_datetime(df_traffy['timestamp'], format='mixed')
df_traffy['date'] = df_traffy['timestamp'].dt.date
df_traffy['date'] = pd.to_datetime(df_traffy['date'])

df_pm['date'] = pd.to_datetime(df_pm['date'])
df_rain['date'] = pd.to_datetime(df_rain['date'])

print("Date conversion complete.")

Date conversion complete.


In [ ]:
# 2. Merging Time-Series Data (PM2.5 & Rainfall)

# Merge Traffy with PM2.5 (Inner Join: Keep only rows with matching dates)
df_merged = pd.merge(df_traffy, df_pm, on='date', how='inner')

# Merge with Rainfall (Inner Join: Keep only rows with matching dates)
df_merged = pd.merge(df_merged, df_rain, on='date', how='inner')

print(f"Shape after merging time-series data: {df_merged.shape}")
print("Columns:", df_merged.columns.tolist())

Shape after merging time-series data: (300000, 32)
Columns: ['ticket_id', 'type', 'organization', 'comment', 'photo', 'photo_after', 'coords', 'address', 'subdistrict', 'district', 'province', 'timestamp', 'state', 'count_reopen', 'last_activity', 'type_clean', 'has_photo', 'comment_len', 'lon', 'lat', 'date', 'Unnamed: 0_x', 'pm25_avg', 'pm25_max', 'pm25_min', 'pm10_avg', 'dust_avg', 'Unnamed: 0_y', 'rainfall_mm', 'rainfall_hours', 'has_rain', 'heavy_rain']


In [ ]:
# 3. Merging Spatial Data (Department)

# Mapping dictionary for District names (English -> Thai)
district_mapping = {
    'Phra Nakhon': 'พระนคร', 'Dusit': 'ดุสิต', 'Nong Chok': 'หนองจอก', 'Bang Rak': 'บางรัก',
    'Bang Khen': 'บางเขน', 'Bang Kapi': 'บางกะปิ', 'Pathum Wan': 'ปทุมวัน', 'Pom Prap Sattru Phai': 'ป้อมปราบศัตรูพ่าย',
    'Phra Khanong': 'พระโขนง', 'Min Buri': 'มีนบุรี', 'Lat Krabang': 'ลาดกระบัง', 'Yan Nawa': 'ยานนาวา',
    'Samphanthawong': 'สัมพันธวงศ์', 'Phaya Thai': 'พญาไท', 'Thon Buri': 'ธนบุรี', 'Bangkok Yai': 'บางกอกใหญ่',
    'Huai Khwang': 'ห้วยขวาง', 'Khlong San': 'คลองสาน', 'Taling Chan': 'ตลิ่งชัน', 'Bangkok Noi': 'บางกอกน้อย',
    'Bang Khun Thian': 'บางขุนเทียน', 'Phasi Charoen': 'ภาษีเจริญ', 'Nong Khaem': 'หนองแขม', 'Rat Burana': 'ราษฎร์บูรณะ',
    'Bang Phlat': 'บางพลัด', 'Din Daeng': 'ดินแดง', 'Bueng Kum': 'บึงกุ่ม', 'Sathon': 'สาทร',
    'Bang Sue': 'บางซื่อ', 'Chatuchak': 'จตุจักร', 'Bang Kho Laem': 'บางคอแหลม', 'Prawet': 'ประเวศ',
    'Khlong Toei': 'คลองเตย', 'Suan Luang': 'สวนหลวง', 'Chom Thong': 'จอมทอง', 'Don Mueang': 'ดอนเมือง',
    'Ratchathewi': 'ราชเทวี', 'Lat Phrao': 'ลาดพร้าว', 'Watthana': 'วัฒนา', 'Bang Khae': 'บางแค',
    'Lak Si': 'หลักสี่', 'Sai Mai': 'สายไหม', 'Khan Na Yao': 'คันนายาว', 'Saphan Sung': 'สะพานสูง',
    'Wang Thonglang': 'วังทองหลาง', 'Khlong Sam Wa': 'คลองสามวา', 'Bang Na': 'บางนา', 'Thawi Watthana': 'ทวีวัฒนา',
    'Thung Khru': 'ทุ่งครุ', 'Bang Bon': 'บางบอน'
}

# Apply mapping to department dataframe
df_dept['district_thai'] = df_dept['district'].map(district_mapping)

# Check for unmapped districts (optional, for debugging)
unmapped = df_dept[df_dept['district_thai'].isna()]['district'].unique()
if len(unmapped) > 0:
    print(f"Warning: Unmapped districts found: {unmapped}")

# Aggregate department count by Thai district name
dept_agg = df_dept.groupby('district_thai').size().reset_index(name='dept_count')

# Merge with main dataframe using the Thai district column (Inner Join)
df_merged = pd.merge(df_merged, dept_agg, left_on='district', right_on='district_thai', how='inner')

# Fill NaN in dept_count with 0 and convert to integer (Should not be needed for inner join, but safe to keep)
df_merged['dept_count'] = df_merged['dept_count'].fillna(0).astype(int)

# Drop the extra 'district_thai' column from merging
df_merged.drop(columns=['district_thai'], inplace=True)

print(f"Shape after merging department data: {df_merged.shape}")
df_merged[['district', 'dept_count']].head()

 'Lam Luk Ka' 'Bang Phli' 'Thanyaburi' 'Mueang Nonthaburi' 'Khlong Luang'
 'Pak Kret']
Shape after merging department data: (300000, 33)
Shape after merging department data: (300000, 33)


,district,dept_count
0,ดินแดง,8
1,มีนบุรี,3
2,ดุสิต,9
3,บางเขน,3
4,จตุจักร,38


In [ ]:
# 4. Feature Selection & Cleaning

# Drop unnecessary columns from merging (like Unnamed indices)
cols_to_drop = [col for col in df_merged.columns if 'Unnamed' in col]
df_final = df_merged.drop(columns=cols_to_drop)
df_final = df_final[df_final['state']=='เสร็จสิ้น']

print("Final columns:", df_final.columns.tolist())

# Save the merged dataset
output_path = "../../data/processed/bangkok_traffy_merged.csv"
# Ensure directory exists
import os
os.makedirs(os.path.dirname(output_path), exist_ok=True)

df_final.to_csv(output_path, index=False)
print(f"Merged data saved to {output_path}")

Final columns: ['ticket_id', 'type', 'organization', 'comment', 'photo', 'photo_after', 'coords', 'address', 'subdistrict', 'district', 'province', 'timestamp', 'state', 'count_reopen', 'last_activity', 'type_clean', 'has_photo', 'comment_len', 'lon', 'lat', 'date', 'pm25_avg', 'pm25_max', 'pm25_min', 'pm10_avg', 'dust_avg', 'rainfall_mm', 'rainfall_hours', 'has_rain', 'heavy_rain', 'dept_count']
 ['ticket_id', 'type', 'organization', 'comment', 'photo', 'photo_after', 'coords', 'address', 'subdistrict', 'district', 'province', 'timestamp', 'state', 'count_reopen', 'last_activity', 'type_clean', 'has_photo', 'comment_len', 'lon', 'lat', 'date', 'pm25_avg', 'pm25_max', 'pm25_min', 'pm10_avg', 'dust_avg', 'rainfall_mm', 'rainfall_hours', 'has_rain', 'heavy_rain', 'dept_count']
Merged data saved to ../../data/processed/bangkok_traffy_merged.csv
Merged data saved to ../../data/processed/bangkok_traffy_merged.csv


In [ ]:
df_final.columns

Index(['ticket_id', 'type', 'organization', 'comment', 'photo', 'photo_after',
       'coords', 'address', 'subdistrict', 'district', 'province', 'timestamp',
       'state', 'count_reopen', 'last_activity', 'type_clean', 'has_photo',
       'comment_len', 'lon', 'lat', 'date', 'pm25_avg', 'pm25_max', 'pm25_min',
       'pm10_avg', 'dust_avg', 'rainfall_mm', 'rainfall_hours', 'has_rain',
       'heavy_rain', 'dept_count'],
      dtype='object')

In [ ]:
df_final.sample(5)

,ticket_id,type,organization,comment,photo,photo_after,coords,address,subdistrict,district,...,pm25_avg,pm25_max,pm25_min,pm10_avg,dust_avg,rainfall_mm,rainfall_hours,has_rain,heavy_rain,dept_count
3169,2024-G22VLN,"{ท่อระบายน้ำ,ความสะอาด}","เขตห้วยขวาง,ฝ่ายรักษาความสะอาดฯ เขตห้วยขวาง",ขยะแป้งหมักที่ไม่ได้ใช้ ถ้าฝนตก มา ขยะมันอ...,https://storage.googleapis.com/traffy_public_b...,https://storage.googleapis.com/traffy_public_b...,"100.57821,13.78148",358/3 ซอย ประชาราษฎร์บำเพ็ญ 7 แยก 6 แขวงห้วยขว...,ห้วยขวาง,ห้วยขวาง,...,17.208333,24.7,11.8,28.304167,0.375000,0.0,0.0,0,0,6
118568,2023-N8ARU9,"{แสงสว่าง,ถนน,ท่อระบายน้ำ}","เขตทุ่งครุ,ฝ่ายโยธา เขตทุ่งครุ",มีการก่อสร้างถนน ช่วยมาติดตั้งไฟส่องสว่างให้หน...,https://storage.googleapis.com/traffy_public_b...,https://storage.googleapis.com/traffy_public_b...,"100.51631,13.60943",เลขที่ 88/68 หมู่ Novaluxx ประชาอุทิศ 131 แขวง...,ทุ่งครุ,ทุ่งครุ,...,34.012500,48.2,17.8,49.012500,0.000000,1.4,8.0,1,0,3
251954,2024-DYMN8B,{ป้าย},"เขตจตุจักร,ฝ่ายเทศกิจ เขตจตุจักร",ป้ายโฆษณาครับ,https://storage.googleapis.com/traffy_public_b...,https://storage.googleapis.com/traffy_public_b...,"100.55809,13.80777",88 Lat Phrao Road Chomphon Khet Chatuchak Bang...,จอมพล,จตุจักร,...,53.779167,84.3,30.9,78.933333,0.000000,0.0,0.0,0,0,38
70062,YTXUFZ,{ถนน},"เขตหลักสี่,สำนักการโยธา กทม.,ฝ่ายโยธา เขตหลักส...",ปัญหา: ภายในซอยดังกล่าว พบมีการขุดเจาะประชาชนไ...,https://storage.googleapis.com/traffy_public_b...,NaN,"100.54865,13.85628",VG4X+HF9 ซอย งามวงศ์วาน 43 แยก 2 แขวงทุ่งสองห้...,ทุ่งสองห้อง,หลักสี่,...,19.541667,25.7,13.6,19.750000,0.000000,0.1,1.0,1,0,8
103490,TKYP79,"{เสียงรบกวน,ถนน}","เขตธนบุรี,ฝ่ายโยธา เขตธนบุรี",ปัญหา : อาคาร เนื่องจาก ปัญหายังไม่ได้รับการแก...,https://storage.googleapis.com/traffy_public_b...,https://storage.googleapis.com/traffy_public_b...,"100.48512,13.71878",โพธิ์นิมิตร ถ. ราชพฤกษ์ แขวงบุคคโล เขตธนบุรี ก...,บุคคโล,ธนบุรี,...,19.587500,29.0,10.2,29.750000,0.916667,5.0,8.0,1,0,8


In [ ]:
df_final.shape

(300000, 31)